In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import shap
import statsmodels.api as sm
import gc

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
individual = ['Member_DOB_Age_Qty',
              'Member_Has_ALS_Diag_Ind','Member_Has_AMI_Diag_Ind',
              'Member_Has_Anxiety_Disorders_Diag_Ind','Member_Has_Asthma_Diag_Ind','Member_Has_Burns_Diag_Ind',
 'Member_Has_Cancer_Diag_Ind','Member_Has_Cardiac_Congenital_Anomalies_Diag_Ind','Member_Has_CHF_Diag_Ind',
 'Member_Has_CAD_Diag_Ind','Member_Has_Cerobrovascular_Disease_Diag_Ind','Member_Has_CIDP_Diag_Ind',
 'Member_Has_CKD_Diag_Ind','Member_Has_Congenital_Hydrocephalus_Diag_Ind',
              'Member_Has_Connective_Tissue_Disorder_Diag_Ind',
              'Member_Has_COPD_Diag_Ind','Member_Has_Crohns_Diag_Ind',
 'Member_Has_Cystic_Fibrosis_Diag_Ind',
 'Member_Has_Depression_Diag_Ind',
 'Member_Has_Dermatomyositis_Diag_Ind',
 'Member_Has_Diabetes_Diag_Ind',
 'Member_Has_Epilepsy_Diag_Ind',
 'Member_Has_ESRD_Diag_Ind',
 'Member_Has_Gaucher_Disease_Diag_Ind',
 'Member_Has_Hemophilia_Diag_Ind',
 'Member_Has_Hepatitis_Diag_Ind',
 'Member_Has_HIV_Diag_Ind',
 'Member_Has_HTN_Diag_Ind',
 'Member_Has_Liver_Disease_Diag_Ind',
 'Member_Has_Lupus_Diag_Ind',
 'Member_Has_Multiple_Sclerosis_Diag_Ind',
 'Member_Has_Myasthenia_Gravis_Diag_Ind',
 'Member_Has_Myositis_Diag_Ind',
 'Member_Has_Nephritis_Nephrosis_or_Sclerosis_Diag_Ind',
 'Member_Has_Osteoarthritis_Diag_Ind',
 'Member_Has_Other_Mental_Health_Diag_Ind',
 'Member_Has_Other_Nerv_Sys_Congenital_Anomalies_Diag_Ind',
 'Member_Has_Parkinsons_Diag_Ind',
 'Member_Has_Peri_Endo_Myocarditis_or_Cardiomyopathy_Diag_Ind',
 'Member_Has_Polymyositis_Diag_Ind',
 'Member_Has_Pulmonary_Heart_Disease_Diag_Ind',
 'Member_Has_RA_Diag_Ind',
 'Member_Has_Resp_Failure_Insuff_Arrest_Diag_Ind',
 'Member_Has_Scleroderma_Diag_Ind',
 'Member_Has_Sickle_Cell_Anemia_Diag_Ind',
 'Member_Has_Spina_Bifida_Diag_Ind',
 'Member_Has_Spinal_Cord_Injury_Diag_Ind',
 'Member_Has_Substance_Abuse_Disorder_Diag_Ind','Member_Has_Ulcerative_Colitis_Diag_Ind', 
                'Member_Is_Exchange_Member_Ind', 
                'Product_Is_ACA_Plan_Ind','Product_Is_Certified_QHP_Ind',
                'Product_Is_Supplemental_Ind',
                'Member_Marital_Status_Current_Divorced',
 'Member_Marital_Status_Current_Married',
 'Member_Marital_Status_Current_Single',
 'Member_Marital_Status_Current_Unknown',
 'Member_Marital_Status_Current_Widow(er)',
 'Product_Metal_Level_Bronze',
 'Product_Metal_Level_Gold',
 'Product_Metal_Level_Not Applicable',
 'Product_Metal_Level_Platinum',
 'Product_Metal_Level_Silver',
 'Product_Type_Not Applicable',
 'Product_Type_Other',
 'Product_Type_POS',
 'Product_Type_PPO',
 'Member_Gender_Current_Female',
 'Race_Black or African American',
 'Race_Hispanic or Latino',
 'Race_Missing',
 'Race_Others',
 'Race_White']

In [3]:
individual_sdoh = ['Member_DOB_Age_Qty',
                 'Member_Has_ALS_Diag_Ind',  'Member_Has_AMI_Diag_Ind',
 'Member_Has_Anxiety_Disorders_Diag_Ind','Member_Has_Asthma_Diag_Ind','Member_Has_Burns_Diag_Ind',
 'Member_Has_Cancer_Diag_Ind','Member_Has_Cardiac_Congenital_Anomalies_Diag_Ind','Member_Has_CHF_Diag_Ind',
 'Member_Has_CAD_Diag_Ind','Member_Has_Cerobrovascular_Disease_Diag_Ind','Member_Has_CIDP_Diag_Ind',
 'Member_Has_CKD_Diag_Ind','Member_Has_Congenital_Hydrocephalus_Diag_Ind',
                    'Member_Has_Connective_Tissue_Disorder_Diag_Ind',
 'Member_Has_COPD_Diag_Ind','Member_Has_Crohns_Diag_Ind',
 'Member_Has_Cystic_Fibrosis_Diag_Ind',
 'Member_Has_Depression_Diag_Ind',
 'Member_Has_Dermatomyositis_Diag_Ind',
 'Member_Has_Diabetes_Diag_Ind',
 'Member_Has_Epilepsy_Diag_Ind',
 'Member_Has_ESRD_Diag_Ind',
 'Member_Has_Gaucher_Disease_Diag_Ind',
 'Member_Has_Hemophilia_Diag_Ind',
 'Member_Has_Hepatitis_Diag_Ind',
 'Member_Has_HIV_Diag_Ind',
 'Member_Has_HTN_Diag_Ind',
 'Member_Has_Liver_Disease_Diag_Ind',
 'Member_Has_Lupus_Diag_Ind',
 'Member_Has_Multiple_Sclerosis_Diag_Ind',
 'Member_Has_Myasthenia_Gravis_Diag_Ind',
 'Member_Has_Myositis_Diag_Ind',
 'Member_Has_Nephritis_Nephrosis_or_Sclerosis_Diag_Ind',
 'Member_Has_Osteoarthritis_Diag_Ind',
 'Member_Has_Other_Mental_Health_Diag_Ind',
 'Member_Has_Other_Nerv_Sys_Congenital_Anomalies_Diag_Ind',
 'Member_Has_Parkinsons_Diag_Ind',
 'Member_Has_Peri_Endo_Myocarditis_or_Cardiomyopathy_Diag_Ind',
 'Member_Has_Polymyositis_Diag_Ind',
 'Member_Has_Pulmonary_Heart_Disease_Diag_Ind',
 'Member_Has_RA_Diag_Ind',
 'Member_Has_Resp_Failure_Insuff_Arrest_Diag_Ind',
 'Member_Has_Scleroderma_Diag_Ind',
 'Member_Has_Sickle_Cell_Anemia_Diag_Ind',
 'Member_Has_Spina_Bifida_Diag_Ind',
 'Member_Has_Spinal_Cord_Injury_Diag_Ind',
 'Member_Has_Substance_Abuse_Disorder_Diag_Ind','Member_Has_Ulcerative_Colitis_Diag_Ind', 
                'Member_Is_Exchange_Member_Ind', 
                'Product_Is_ACA_Plan_Ind','Product_Is_Certified_QHP_Ind',
                'Product_Is_Supplemental_Ind',
 'ACS_PCT_ENGL_NOT_ALL',
 'ACS_PCT_FEMALE',
 'ACS_PCT_FOREIGN_BORN',
 'ACS_PCT_LIMIT_ENGLISH',
 'ACS_PCT_NON_CITIZEN',
 'ACS_PCT_1UP_PERS_1ROOM',
 'ACS_PCT_AGE_65UP',
 'ACS_PCT_ASIAN',
 'ACS_PCT_BLACK',
 'ACS_PCT_HISPAN',
 'ACS_PCT_WHITE',
 'ACS_PCT_DISABLE',
 'ACS_HH_SIZE',
 'ACS_PCT_DIVORCE_SEPARAT',
 'ACS_PCT_HH_NO_FUEL',
 'ACS_PCT_ADMIN',
 'ACS_PCT_CONSTRUCT',
 'ACS_PCT_EDUC',
 'ACS_PCT_FINANCE',
 'ACS_PCT_MANUFACT',
 'ACS_PCT_NATURE',
 'ACS_PCT_PROFESS',
 'ACS_PCT_RETAIL',
 'ACS_PCT_TRANSPORT',
 'ACS_PCT_UNEMPLOY',
 'ACS_PCT_NONVA_POOR',
 'ACS_GINI_INDEX',
 'ACS_MEDIAN_HH_INCOME',
 'ACS_PCT_FOOD_STAMP',
 'ACS_PCT_HH_PUB_ASSIST',
 'ACS_PCT_BACHELOR_DGR',
 'ACS_PCT_LT_HS',
 'ACS_PCT_GRP_QRT',
 'CCBP_RATE_CS_PER_1000',
 'ACS_PCT_NO_VEH',
 'ACS_PCT_MEDICAID_ANY64',
 'ACS_PCT_PRIVATE_ANY64',
 'ACS_PCT_PRIVATE_EMPL64',
 'ACS_PCT_UNINSURED64',
 'ACS_PCT_UNINSURED','any_open_park','count_open_parks',
 'count_tri_facilities',
                   'Member_Marital_Status_Current_Unknown',
                   'Member_Marital_Status_Current_Divorced',
 'Member_Marital_Status_Current_Married',
 'Member_Marital_Status_Current_Single',
 'Member_Marital_Status_Current_Widow(er)',
 'Product_Metal_Level_Bronze',
 'Product_Metal_Level_Gold',
 'Product_Metal_Level_Platinum',
 'Product_Metal_Level_Silver',
                   'Product_Metal_Level_Not Applicable',
  'Product_Type_Not Applicable',
 'Product_Type_Other',
 'Product_Type_POS',
 'Product_Type_PPO',
 'Member_Gender_Current_Female',
 'Race_Black or African American',
 'Race_Hispanic or Latino',
 'Race_Missing',
 'Race_Others',
 'Race_White']

In [4]:
individual_sdi = ['Member_DOB_Age_Qty','Member_Has_ALS_Diag_Ind','Member_Has_AMI_Diag_Ind',
              'Member_Has_Anxiety_Disorders_Diag_Ind','Member_Has_Asthma_Diag_Ind','Member_Has_Burns_Diag_Ind',
 'Member_Has_Cancer_Diag_Ind','Member_Has_Cardiac_Congenital_Anomalies_Diag_Ind','Member_Has_CHF_Diag_Ind',
 'Member_Has_CAD_Diag_Ind','Member_Has_Cerobrovascular_Disease_Diag_Ind','Member_Has_CIDP_Diag_Ind',
 'Member_Has_CKD_Diag_Ind','Member_Has_Congenital_Hydrocephalus_Diag_Ind',
              'Member_Has_Connective_Tissue_Disorder_Diag_Ind',
              'Member_Has_COPD_Diag_Ind','Member_Has_Crohns_Diag_Ind',
 'Member_Has_Cystic_Fibrosis_Diag_Ind',
 'Member_Has_Depression_Diag_Ind',
 'Member_Has_Dermatomyositis_Diag_Ind',
 'Member_Has_Diabetes_Diag_Ind',
 'Member_Has_Epilepsy_Diag_Ind',
 'Member_Has_ESRD_Diag_Ind',
 'Member_Has_Gaucher_Disease_Diag_Ind',
 'Member_Has_Hemophilia_Diag_Ind',
 'Member_Has_Hepatitis_Diag_Ind',
 'Member_Has_HIV_Diag_Ind',
 'Member_Has_HTN_Diag_Ind',
 'Member_Has_Liver_Disease_Diag_Ind',
 'Member_Has_Lupus_Diag_Ind',
 'Member_Has_Multiple_Sclerosis_Diag_Ind',
 'Member_Has_Myasthenia_Gravis_Diag_Ind',
 'Member_Has_Myositis_Diag_Ind',
 'Member_Has_Nephritis_Nephrosis_or_Sclerosis_Diag_Ind',
 'Member_Has_Osteoarthritis_Diag_Ind',
 'Member_Has_Other_Mental_Health_Diag_Ind',
 'Member_Has_Other_Nerv_Sys_Congenital_Anomalies_Diag_Ind',
 'Member_Has_Parkinsons_Diag_Ind',
 'Member_Has_Peri_Endo_Myocarditis_or_Cardiomyopathy_Diag_Ind',
 'Member_Has_Polymyositis_Diag_Ind',
 'Member_Has_Pulmonary_Heart_Disease_Diag_Ind',
 'Member_Has_RA_Diag_Ind',
 'Member_Has_Resp_Failure_Insuff_Arrest_Diag_Ind',
 'Member_Has_Scleroderma_Diag_Ind',
 'Member_Has_Sickle_Cell_Anemia_Diag_Ind',
 'Member_Has_Spina_Bifida_Diag_Ind',
 'Member_Has_Spinal_Cord_Injury_Diag_Ind',
 'Member_Has_Substance_Abuse_Disorder_Diag_Ind','Member_Has_Ulcerative_Colitis_Diag_Ind', 
                'Member_Is_Exchange_Member_Ind', 
                'Product_Is_ACA_Plan_Ind','Product_Is_Certified_QHP_Ind',
                'Product_Is_Supplemental_Ind',
                'Member_Marital_Status_Current_Divorced',
 'Member_Marital_Status_Current_Married',
 'Member_Marital_Status_Current_Single',
 'Member_Marital_Status_Current_Unknown',
 'Member_Marital_Status_Current_Widow(er)',
 'Product_Metal_Level_Bronze',
 'Product_Metal_Level_Gold',
 'Product_Metal_Level_Not Applicable',
 'Product_Metal_Level_Platinum',
 'Product_Metal_Level_Silver',
  'Product_Type_Not Applicable',
 'Product_Type_Other',
 'Product_Type_POS',
 'Product_Type_PPO',
 'Member_Gender_Current_Female',
 'Race_Black or African American',
 'Race_Hispanic or Latino',
 'Race_Missing',
 'Race_Others',
 'Race_White',
                  'sdi']

In [16]:
MODEL_PATH = "../results/poisson_intensity_sample/poisson_pcp_count_individual_sdoh_sample.pkl"
# MODEL_PATH = "../results/xgboost_poisson_full/xgb_poisson_er_count_individual_sdoh_full.pkl"
# Outcome for Poisson count model
OUTCOME_COL = "Office_PCP_Count"
OUTCOME_NAME = "pcp_count"

# Feature-set name
FEATURE_SET_NAME = "individual_sdoh"

RANDOM_STATE = 42

TOP_N_TOTAL_FEATURES = 20
TOP_N_NSDOH_FOR_PFA = 10
TOP_N_INDIVIDUAL_FOR_FCI = 18

In [17]:
x_train = pd.read_csv('../data_processed/x_train_nb_sample.csv')
# x_train = pd.read_csv('../../../xc2440/SDoH_insurance/data_processed/x_train.csv')

analysis2018_raw = pd.read_csv(
    '../../../xc2440/SDoH_insurance/data_processed/analysis2018_raw.csv',
    usecols=["Member_DOB_Age_Qty", OUTCOME_COL],
    low_memory=False
)

In [18]:
sample_frac = 0.1
seed = 42

# Load only one column just to recreate the full dataframe index
x_train_index_helper = pd.read_csv(
    '../../../xc2440/SDoH_insurance/data_processed/x_train.csv',
    usecols=["Member_DOB_Age_Qty"]
)

sample_idx = x_train_index_helper.sample(
    frac=sample_frac,
    random_state=seed
).index

del x_train_index_helper
gc.collect()

print(len(sample_idx))
print(sample_idx[:10])

1476720
Int64Index([14001114,  7724655,  7091627,  5698352, 12010168,  8138818,
             8546822, 12613390, 13331793, 10536510],
           dtype='int64')


In [19]:
analysis2018_raw_sample = analysis2018_raw.loc[sample_idx].reset_index(drop=True)

del analysis2018_raw
gc.collect()

y_count_sample = analysis2018_raw_sample[OUTCOME_COL].astype(int)
# y_count = analysis2018_raw[OUTCOME_COL].astype(int)

print("analysis2018_raw:", analysis2018_raw_sample.shape)
print("y_count:", y_count_sample.shape)

print("Outcome distribution:")
print(y_count_sample.value_counts().sort_index().head(30))

print("Mean:", y_count_sample.mean())
print("Variance:", y_count_sample.var())

analysis2018_raw: (1476720, 2)
y_count: (1476720,)
Outcome distribution:
0     1306690
1      148377
2       18018
3        2796
4         596
5         152
6          38
7          17
8          12
9          11
10          1
11          2
12          3
13          1
14          1
15          2
19          2
23          1
Name: Office_PCP_Count, dtype: int64
Mean: 0.13318232298607724
Variance: 0.1621538241173352


In [20]:
nsdoh_cols = [
    col for col in individual_sdoh
    if (
        col.startswith("ACS_")
        or col.startswith("CCBP_")
        or col in ["any_open_park", "count_open_parks", "count_tri_facilities"]
    )
]

individual_cols = [
    col for col in individual_sdoh
    if col not in nsdoh_cols
]

print("Total individual + SDoH features:", len(individual_sdoh))
print("Individual columns:", len(individual_cols))
print("nSDoH columns:", len(nsdoh_cols))

print(nsdoh_cols)

Total individual + SDoH features: 116
Individual columns: 73
nSDoH columns: 43
['ACS_PCT_ENGL_NOT_ALL', 'ACS_PCT_FEMALE', 'ACS_PCT_FOREIGN_BORN', 'ACS_PCT_LIMIT_ENGLISH', 'ACS_PCT_NON_CITIZEN', 'ACS_PCT_1UP_PERS_1ROOM', 'ACS_PCT_AGE_65UP', 'ACS_PCT_ASIAN', 'ACS_PCT_BLACK', 'ACS_PCT_HISPAN', 'ACS_PCT_WHITE', 'ACS_PCT_DISABLE', 'ACS_HH_SIZE', 'ACS_PCT_DIVORCE_SEPARAT', 'ACS_PCT_HH_NO_FUEL', 'ACS_PCT_ADMIN', 'ACS_PCT_CONSTRUCT', 'ACS_PCT_EDUC', 'ACS_PCT_FINANCE', 'ACS_PCT_MANUFACT', 'ACS_PCT_NATURE', 'ACS_PCT_PROFESS', 'ACS_PCT_RETAIL', 'ACS_PCT_TRANSPORT', 'ACS_PCT_UNEMPLOY', 'ACS_PCT_NONVA_POOR', 'ACS_GINI_INDEX', 'ACS_MEDIAN_HH_INCOME', 'ACS_PCT_FOOD_STAMP', 'ACS_PCT_HH_PUB_ASSIST', 'ACS_PCT_BACHELOR_DGR', 'ACS_PCT_LT_HS', 'ACS_PCT_GRP_QRT', 'CCBP_RATE_CS_PER_1000', 'ACS_PCT_NO_VEH', 'ACS_PCT_MEDICAID_ANY64', 'ACS_PCT_PRIVATE_ANY64', 'ACS_PCT_PRIVATE_EMPL64', 'ACS_PCT_UNINSURED64', 'ACS_PCT_UNINSURED', 'any_open_park', 'count_open_parks', 'count_tri_facilities']


In [21]:
print("Loading Poisson model...")

try:
    poisson_result = load_pickle(MODEL_PATH)
except:
    poisson_result = sm.load(MODEL_PATH)

print("Model loaded.")
print(type(poisson_result))

Loading Poisson model...
Model loaded.
<class 'statsmodels.genmod.generalized_linear_model.GLMResultsWrapper'>


In [22]:
# For sample
coef_df = pd.DataFrame({
    "feature": poisson_result.params.index,
    "coef": poisson_result.params.values,
    "IRR": np.exp(poisson_result.params.values),
    "pvalue": poisson_result.pvalues.values
})

# z-score = coefficient / standard error
coef_df["std_error"] = poisson_result.bse.values
coef_df["z_score"] = coef_df["coef"] / coef_df["std_error"]
coef_df["abs_z_score"] = coef_df["z_score"].abs()
coef_df["abs_coef"] = coef_df["coef"].abs()

# Remove intercept
coef_df = coef_df[coef_df["feature"] != "const"].copy()

# Keep only real feature columns present in x_train
coef_df = coef_df[coef_df["feature"].isin(x_train.columns)].copy()

coef_df = coef_df.sort_values("abs_z_score", ascending=False)

coef_df.head(30)
# For XGBoost Poisson
# XGBoost Booster does not have params / pvalues / bse
# Use model feature importance instead

# importance_dict = poisson_result.get_score(importance_type="total_gain")

# model_features = poisson_result.feature_names

# coef_df = pd.DataFrame({
#     "feature": model_features,
#     "importance": [
#         importance_dict.get(f, 0) for f in model_features
#     ]
# })

# # Keep only real feature columns present in x_train
# coef_df = coef_df[coef_df["feature"].isin(x_train.columns)].copy()

# coef_df = coef_df.sort_values("importance", ascending=False)

# coef_df.head(30)

,feature,coef,IRR,pvalue,std_error,z_score,abs_z_score,abs_coef
28,Member_Has_HTN_Diag_Ind,1.741546,5.706158,0.000000e+00,0.007010,248.451519,248.451519,1.741546
36,Member_Has_Other_Mental_Health_Diag_Ind,0.874226,2.397019,0.000000e+00,0.007843,111.470271,111.470271,0.874226
1,Member_DOB_Age_Qty,-0.288980,0.749027,0.000000e+00,0.002828,-102.199415,102.199415,0.288980
4,Member_Has_Anxiety_Disorders_Diag_Ind,0.605524,1.832213,0.000000e+00,0.008938,67.750248,67.750248,0.605524
21,Member_Has_Diabetes_Diag_Ind,0.536905,1.710703,0.000000e+00,0.008209,65.404031,65.404031,0.536905
5,Member_Has_Asthma_Diag_Ind,0.756671,2.131169,0.000000e+00,0.014220,53.211490,53.211490,0.756671
112,Race_White,0.175988,1.192423,3.156139e-232,0.005409,32.537402,32.537402,0.175988
35,Member_Has_Osteoarthritis_Diag_Ind,0.365739,1.441579,1.994889e-220,0.011540,31.692121,31.692121,0.365739
99,Member_Marital_Status_Current_Single,-0.219805,0.802675,9.218324e-208,0.007146,-30.759177,30.759177,0.219805
107,Product_Type_PPO,0.138163,1.148163,1.507007e-125,0.005797,23.833354,23.833354,0.138163


In [23]:
top_features = coef_df.head(TOP_N_TOTAL_FEATURES)["feature"].tolist()

top_individual_features = [
    col for col in top_features
    if col not in nsdoh_cols
]

top_nsdoh_features = [
    col for col in top_features
    if col in nsdoh_cols
]

# If not enough nSDoH variables appear in top 20,
# take top nSDoH variables separately from the model importance table
if len(top_nsdoh_features) < 3:
    top_nsdoh_features = (
        coef_df[coef_df["feature"].isin(nsdoh_cols)]
        .head(TOP_N_NSDOH_FOR_PFA)["feature"]
        .tolist()
    )

# Limit individual variables for FCI
top_individual_features = top_individual_features[:TOP_N_INDIVIDUAL_FOR_FCI]

print("Top individual features for FCI:")
print(top_individual_features)

print("\nTop nSDoH features for PFA:")
print(top_nsdoh_features)
# top_features = coef_df.head(TOP_N_TOTAL_FEATURES)["feature"].tolist()

# top_individual_features = [
#     col for col in top_features
#     if col not in nsdoh_cols
# ]

# top_nsdoh_features = [
#     col for col in top_features
#     if col in nsdoh_cols
# ]

# # If not enough nSDoH variables appear in top features,
# # take top nSDoH variables separately from the XGBoost importance table
# if len(top_nsdoh_features) < 3:
#     top_nsdoh_features = (
#         coef_df[coef_df["feature"].isin(nsdoh_cols)]
#         .sort_values("importance", ascending=False)
#         .head(TOP_N_NSDOH_FOR_PFA)["feature"]
#         .tolist()
#     )

# # Limit individual variables for FCI
# top_individual_features = top_individual_features[:TOP_N_INDIVIDUAL_FOR_FCI]

# print("Top individual features for FCI:")
# print(top_individual_features)

# print("\nTop nSDoH features for PFA:")
# print(top_nsdoh_features)

Top individual features for FCI:
['Member_Has_HTN_Diag_Ind', 'Member_Has_Other_Mental_Health_Diag_Ind', 'Member_DOB_Age_Qty', 'Member_Has_Anxiety_Disorders_Diag_Ind', 'Member_Has_Diabetes_Diag_Ind', 'Member_Has_Asthma_Diag_Ind', 'Race_White', 'Member_Has_Osteoarthritis_Diag_Ind', 'Member_Marital_Status_Current_Single', 'Product_Type_PPO', 'Member_Has_Liver_Disease_Diag_Ind', 'Member_Gender_Current_Female', 'Product_Type_POS', 'Member_Has_COPD_Diag_Ind', 'Member_Has_Depression_Diag_Ind', 'Member_Marital_Status_Current_Married', 'Member_Has_Burns_Diag_Ind', 'Member_Has_Resp_Failure_Insuff_Arrest_Diag_Ind']

Top nSDoH features for PFA:
['ACS_PCT_HISPAN', 'ACS_PCT_NATURE', 'ACS_PCT_TRANSPORT', 'count_tri_facilities', 'ACS_PCT_PROFESS', 'ACS_PCT_RETAIL', 'ACS_PCT_NO_VEH', 'ACS_PCT_BACHELOR_DGR', 'ACS_MEDIAN_HH_INCOME', 'ACS_PCT_ENGL_NOT_ALL']


In [12]:
x_nsdoh = x_train[top_nsdoh_features].copy()

x_nsdoh = x_nsdoh.replace([np.inf, -np.inf], np.nan)
x_nsdoh = x_nsdoh.fillna(x_nsdoh.median())

scaler = StandardScaler()
x_nsdoh_scaled = scaler.fit_transform(x_nsdoh)

pfa = FactorAnalysis(
    n_components=1,
    random_state=RANDOM_STATE
)

nsdoh_pfa_score = pfa.fit_transform(x_nsdoh_scaled).ravel()

# Standardize score
nsdoh_pfa_score = (
    nsdoh_pfa_score - nsdoh_pfa_score.mean()
) / nsdoh_pfa_score.std()

In [13]:
pfa_loadings = pd.DataFrame({
    "feature": top_nsdoh_features,
    "loading": pfa.components_[0]
})

pfa_loadings["abs_loading"] = pfa_loadings["loading"].abs()

bad_direction_cols = [
    "ACS_PCT_UNEMPLOY",
    "ACS_PCT_NONVA_POOR",
    "ACS_PCT_FOOD_STAMP",
    "ACS_PCT_HH_PUB_ASSIST",
    "ACS_PCT_LT_HS",
    "ACS_PCT_NO_VEH",
    "ACS_PCT_UNINSURED64",
    "ACS_PCT_UNINSURED",
    "ACS_PCT_1UP_PERS_1ROOM",
    "ACS_PCT_LIMIT_ENGLISH",
    "ACS_PCT_NON_CITIZEN",
    "ACS_GINI_INDEX"
]

bad_present = [
    c for c in bad_direction_cols
    if c in pfa_loadings["feature"].values
]

if len(bad_present) > 0:
    
    mean_bad_loading = pfa_loadings.loc[
        pfa_loadings["feature"].isin(bad_present),
        "loading"
    ].mean()
    
    print("Mean loading for disadvantage-like columns:", mean_bad_loading)
    
    if mean_bad_loading < 0:
        print("Flipping PFA score so higher = more disadvantage.")
        nsdoh_pfa_score = -nsdoh_pfa_score
        pfa_loadings["loading"] = -pfa_loadings["loading"]

else:
    
    if "ACS_MEDIAN_HH_INCOME" in pfa_loadings["feature"].values:
        
        income_loading = pfa_loadings.loc[
            pfa_loadings["feature"] == "ACS_MEDIAN_HH_INCOME",
            "loading"
        ].iloc[0]
        
        print("Income loading:", income_loading)
        
        if income_loading > 0:
            print("Flipping PFA score because income positive means higher score may reflect advantage.")
            nsdoh_pfa_score = -nsdoh_pfa_score
            pfa_loadings["loading"] = -pfa_loadings["loading"]

pfa_loadings = pfa_loadings.sort_values("abs_loading", ascending=False)

pfa_loadings

Mean loading for disadvantage-like columns: 0.4391577270050661


,feature,loading,abs_loading
7,ACS_PCT_LIMIT_ENGLISH,0.879402,0.879402
10,ACS_PCT_ENGL_NOT_ALL,0.686550,0.686550
1,ACS_PCT_EDUC,-0.252700,0.252700
9,ACS_PCT_TRANSPORT,0.186301,0.186301
3,ACS_PCT_PRIVATE_EMPL64,-0.091632,0.091632
6,ACS_PCT_HH_NO_FUEL,0.087414,0.087414
8,count_open_parks,0.069341,0.069341
4,ACS_PCT_NATURE,-0.058093,0.058093
5,ACS_PCT_FEMALE,-0.041978,0.041978
0,count_tri_facilities,-0.017374,0.017374


In [ ]:
df = x_train.copy()

df[f"{OUTCOME_NAME}_observed"] = y_count_sample.values
# df[f"{OUTCOME_NAME}_observed"] = y_count.values
df[f"{OUTCOME_NAME}_log"] = np.log1p(df[f"{OUTCOME_NAME}_observed"])
df["age_raw"] = analysis2018_raw_sample["Member_DOB_Age_Qty"].values
# df["age_raw"] = analysis2018_raw["Member_DOB_Age_Qty"].values
df["age_65_group"] = (df["age_raw"] >= 65).astype(int)

df[f"{OUTCOME_NAME}_nSDoH_pfa_score"] = nsdoh_pfa_score


# nsdoh_threshold_85 = df[f"{OUTCOME_NAME}_nSDoH_pfa_score"].quantile(0.85)

# df["nsdoh_85_bin"] = (
#     df[f"{OUTCOME_NAME}_nSDoH_pfa_score"] >= nsdoh_threshold_85
# ).astype(int)

# print("nSDoH 85th percentile threshold:", nsdoh_threshold_85)
# print(df["nsdoh_85_bin"].value_counts())
# print(df["nsdoh_85_bin"].mean())

In [ ]:
pfa_score_path = os.path.join(
    '../results/poisson_intensity_sample',
    f"{OUTCOME_NAME}_{FEATURE_SET_NAME}_pfa_scores_fullbin.csv"
)
# pfa_score_path = os.path.join(
#     '../results/xgboost_poisson_full',
#     f"xgb_{OUTCOME_NAME}_{FEATURE_SET_NAME}_pfa_scores_fullbin.csv"
# )


df[
    [
        f"{OUTCOME_NAME}_observed",
        f"{OUTCOME_NAME}_nSDoH_pfa_score",
#         "nsdoh_85_bin",
        "age_raw",
        "age_65_group"
    ]
].to_csv(pfa_score_path, index=False)

print("Saved:", pfa_score_path)

Saved: ../results/xgboost_poisson_full/xgb_er_count_individual_sdoh_pfa_scores_fullbin.csv


In [16]:
# Use count outcome as the FCI outcome
causal_cols = [
    f"{OUTCOME_NAME}_log",
#     "nsdoh_85_bin",
    f"{OUTCOME_NAME}_nSDoH_pfa_score",
    "age_65_group"
]

# Do not include raw age if we already included age_65_group
top_individual_features_final = [
    col for col in top_individual_features
    if col != "Member_DOB_Age_Qty"
]

causal_cols += top_individual_features_final

# remove duplicate columns while preserving order
causal_cols = list(dict.fromkeys(causal_cols))

poisson_causal_dat = df[causal_cols].copy()

poisson_causal_dat = poisson_causal_dat.replace([np.inf, -np.inf], np.nan)
poisson_causal_dat = poisson_causal_dat.fillna(0)

# Ensure numeric
for col in poisson_causal_dat.columns:
    poisson_causal_dat[col] = pd.to_numeric(poisson_causal_dat[col], errors="coerce")

poisson_causal_dat = poisson_causal_dat.fillna(0)

print("Causal data shape:", poisson_causal_dat.shape)
print(poisson_causal_dat.head())
print(poisson_causal_dat.nunique().sort_values())

Causal data shape: (14767202, 12)
   er_count_log  er_count_nSDoH_pfa_score  age_65_group  \
0           0.0                  0.403263             0   
1           0.0                 -0.494809             0   
2           0.0                 -0.482312             0   
3           0.0                 -0.365236             0   
4           0.0                  0.021318             0   

   Member_Has_Gaucher_Disease_Diag_Ind  Product_Metal_Level_Silver  \
0                                    0                           1   
1                                    0                           0   
2                                    0                           1   
3                                    0                           1   
4                                    0                           0   

   Product_Type_POS  Member_Has_Substance_Abuse_Disorder_Diag_Ind  \
0                 0                                             0   
1                 0                                 

In [17]:
constant_cols = [
    col for col in poisson_causal_dat.columns
    if poisson_causal_dat[col].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_cols)

poisson_causal_dat = poisson_causal_dat.drop(columns=constant_cols)

print("Final shape after dropping constants:", poisson_causal_dat.shape)

Constant columns: []
Final shape after dropping constants: (14767202, 12)


In [18]:
# full_output_path = os.path.join(
#     '../results/poisson_intensity_sample',
#     f"{OUTCOME_NAME}_{FEATURE_SET_NAME}_poisson_causal_dat_nsdoh_pfa_full.csv"
# )
full_output_path = os.path.join(
    '../results/xgboost_poisson_full',
    f"xgb_{OUTCOME_NAME}_{FEATURE_SET_NAME}_poisson_causal_dat_nsdoh_pfa_full.csv"
)

poisson_causal_dat.to_csv(full_output_path, index=False)

print("Saved full causal data:", full_output_path)

Saved full causal data: ../results/xgboost_poisson_full/xgb_er_count_individual_sdoh_poisson_causal_dat_nsdoh_pfa_full.csv


In [26]:
full_output_path = os.path.join(
    "../results/poisson_intensity_sample",
    f"{OUTCOME_NAME}_{FEATURE_SET_NAME}_poisson_causal_dat_nsdoh_pfa_full.csv"
)
poisson_causal_dat = pd.read_csv(full_output_path)

FCI_SAMPLE_FRAC = 0.01

poisson_causal_sample = poisson_causal_dat.sample(
    frac=FCI_SAMPLE_FRAC,
    random_state=RANDOM_STATE
)

sample_output_path = os.path.join(
    "../results/poisson_intensity_sample",
    f"{OUTCOME_NAME}_{FEATURE_SET_NAME}_poisson_causal_dat_log_nsdoh85_1pct.csv"
)

poisson_causal_sample.to_csv(sample_output_path, index=False)

print("Saved sample:", sample_output_path)
print("Sample shape:", poisson_causal_sample.shape)

Saved sample: ../results/poisson_intensity_sample/er_count_individual_sdoh_poisson_causal_dat_log_nsdoh85_1pct.csv
Sample shape: (14767, 20)


In [21]:
print("Outcome summary:")
print(poisson_causal_dat[f"{OUTCOME_NAME}_log"].describe())

print("\nOutcome distribution:")
print(poisson_causal_dat[f"{OUTCOME_NAME}_log"].value_counts().sort_index().head(30))

print("\nnSDoH 85 distribution:")
print(poisson_causal_dat[f"{OUTCOME_NAME}_nSDoH_pfa_score"].value_counts(normalize=True))

print("\nColumns for FCI:")
print(poisson_causal_dat.columns.tolist())

Outcome summary:
count    1.476720e+06
mean     8.664179e-02
std      2.465758e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.178054e+00
Name: pcp_count_log, dtype: float64

Outcome distribution:
0.000000    1306690
0.693147     148377
1.098612      18018
1.386294       2796
1.609438        596
1.791759        152
1.945910         38
2.079442         17
2.197225         12
2.302585         11
2.397895          1
2.484907          2
2.564949          3
2.639057          1
2.708050          1
2.772589          2
2.995732          2
3.178054          1
Name: pcp_count_log, dtype: int64

nSDoH 85 distribution:
 0.153406    1.612289e-02
-1.176016    1.488231e-02
-1.735827    1.396609e-02
 0.214116    1.308305e-02
 0.048030    1.278441e-02
                 ...     
 0.822095    1.354353e-05
 1.837773    1.083482e-05
-5.039541    7.448941e-06
 0.194721    6.771764e-07
-7.112923    6.771764e-07
Name: pcp_count_nSDoH_pfa_score, Length: 506

In [22]:
print(poisson_causal_dat.isna().sum().sum())
print(np.isinf(poisson_causal_dat.values).sum())
print(poisson_causal_dat.dtypes)

0
0
pcp_count_log                                     float64
pcp_count_nSDoH_pfa_score                         float64
age_65_group                                        int64
Member_Has_HTN_Diag_Ind                             int64
Member_Has_Other_Mental_Health_Diag_Ind             int64
Member_Has_Anxiety_Disorders_Diag_Ind               int64
Member_Has_Diabetes_Diag_Ind                        int64
Member_Has_Asthma_Diag_Ind                          int64
Race_White                                          int64
Member_Has_Osteoarthritis_Diag_Ind                  int64
Member_Marital_Status_Current_Single                int64
Product_Type_PPO                                    int64
Member_Has_Liver_Disease_Diag_Ind                   int64
Member_Gender_Current_Female                        int64
Product_Type_POS                                    int64
Member_Has_COPD_Diag_Ind                            int64
Member_Has_Depression_Diag_Ind                      int64
Member_Mar

In [29]:
poisson_causal_dat.groupby("nsdoh_85_bin")["er_count_log"].describe()

,count,mean,std,min,25%,50%,75%,max
nsdoh_85_bin,,,,,,,,
0,1254432.0,0.010648,0.089214,0.0,0.0,0.0,0.0,2.079442
1,222288.0,0.013354,0.100258,0.0,0.0,0.0,0.0,2.079442


In [30]:
print("nSDoH 85 distribution in full data:")
print(poisson_causal_dat["nsdoh_85_bin"].value_counts(normalize=True))

print("\nnSDoH 85 distribution in 1% sample:")
print(poisson_causal_sample["nsdoh_85_bin"].value_counts(normalize=True))

nSDoH 85 distribution in full data:
0    0.849472
1    0.150528
Name: nsdoh_85_bin, dtype: float64

nSDoH 85 distribution in 1% sample:
0    0.851493
1    0.148507
Name: nsdoh_85_bin, dtype: float64


In [31]:
if f"{OUTCOME_NAME}_log" in poisson_causal_sample.columns:
    print("Log outcome summary in 1% sample:")
    print(poisson_causal_sample[f"{OUTCOME_NAME}_log"].describe())

Log outcome summary in 1% sample:
count    14767.000000
mean         0.011450
std          0.092157
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.609438
Name: er_count_log, dtype: float64
